In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "a3048f6b",
   "metadata": {},
   "source": [
    "# 📊 Module 10: Model Evaluation & Benchmarking\n",
    "**Movie Recommendation System Using Machine Learning**\n",
    "\n",
    "---\n",
    "\n",
    "### 🎯 Objectives\n",
    "1. Understand why recommender system evaluation requires both **Rating Accuracy** and **Ranking Quality** metrics.\n",
    "2. Calculate and explain **RMSE** (Root Mean Squared Error) and **MAE** (Mean Absolute Error).\n",
    "3. Calculate and explain **Precision@K** and **Recall@K** using a relevance threshold ($r \\ge 3.5$).\n",
    "4. Calculate **Catalog Coverage** to measure recommendation diversity.\n",
    "5. Benchmark Item-Based CF against User-Based CF.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "82da21d6",
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import sys\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "sys.path.insert(0, \"..\")\n",
    "from src.recommender import MovieRecommender\n",
    "from src.evaluation import run_full_evaluation\n",
    "\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "plt.rcParams[\"figure.figsize\"] = (10, 5)\n",
    "\n",
    "print(\"Environment ready!\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2fd4b6ad",
   "metadata": {},
   "source": [
    "## 1. Mathematical Formulas for Metrics\n",
    "\n",
    "### 1. Root Mean Squared Error (RMSE)\n",
    "$$\\text{RMSE} = \\sqrt{\\frac{1}{|T|} \\sum_{(u,i) \\in T} (r_{u,i} - \\hat{r}_{u,i})^2}$$\n",
    "Penalizes large errors heavily.\n",
    "\n",
    "### 2. Mean Absolute Error (MAE)\n",
    "$$\\text{MAE} = \\frac{1}{|T|} \\sum_{(u,i) \\in T} |r_{u,i} - \\hat{r}_{u,i}|$$\n",
    "Linear, interpretable average star error.\n",
    "\n",
    "### 3. Precision@K and Recall@K\n",
    "- Item is *Relevant* if actual rating $\\ge 3.5$.\n",
    "$$\\text{Precision@K} = \\frac{|\\text{Recommended}_K \\cap \\text{Relevant}|}{K}$$\n",
    "$$\\text{Recall@K} = \\frac{|\\text{Recommended}_K \\cap \\text{Relevant}|}{|\\text{Relevant}|}$$\n",
    "\n",
    "### 4. Catalog Coverage\n",
    "$$\\text{Coverage} = \\frac{|\\bigcup_u \\text{Recommended}_u|}{|\\text{Catalog Items}|} \\times 100\\%$$\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6b5a8d47",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load model and test ratings\n",
    "model = MovieRecommender.load(\"../models/recommender_model.joblib\")\n",
    "test_ratings = pd.read_csv(\"../dataset/test_ratings.csv\")\n",
    "\n",
    "print(f\"Loaded {len(test_ratings):,} held-out test ratings.\")\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9b44f08e",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run complete benchmark\n",
    "results = run_full_evaluation(\n",
    "    recommender=model,\n",
    "    test_ratings_df=test_ratings,\n",
    "    k=10,\n",
    "    relevance_threshold=3.5,\n",
    "    num_eval_users=80,\n",
    "    max_rating_pairs=2000\n",
    ")\n",
    "\n",
    "results_df = pd.DataFrame(results).T\n",
    "display(results_df)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "aa6eb229",
   "metadata": {},
   "source": [
    "## 2. Visualization of Evaluation Benchmark"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "fa28dac9",
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))\n",
    "\n",
    "# RMSE & MAE\n",
    "models = [\"Item-Based CF\", \"User-Based CF\"]\n",
    "rmse = [results[\"item_based\"][\"RMSE\"], results[\"user_based\"][\"RMSE\"]]\n",
    "mae = [results[\"item_based\"][\"MAE\"], results[\"user_based\"][\"MAE\"]]\n",
    "\n",
    "x = np.arange(len(models))\n",
    "w = 0.35\n",
    "ax1.bar(x - w/2, rmse, w, label=\"RMSE\", color=\"#4C72B0\")\n",
    "ax1.bar(x + w/2, mae, w, label=\"MAE\", color=\"#55A868\")\n",
    "ax1.set_title(\"Rating Prediction Error (Lower is Better)\")\n",
    "ax1.set_ylabel(\"Error (Stars)\")\n",
    "ax1.set_xticks(x)\n",
    "ax1.set_xticklabels(models)\n",
    "ax1.legend()\n",
    "\n",
    "# Coverage\n",
    "cov = [results[\"item_based\"][\"Coverage@10 (%)\"], results[\"user_based\"][\"Coverage@10 (%)\"]]\n",
    "ax2.bar(models, cov, color=[\"#C44E52\", \"#8172B2\"])\n",
    "ax2.set_title(\"Catalog Coverage @ 10 (Higher is Better)\")\n",
    "ax2.set_ylabel(\"Catalog Coverage (%)\")\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "56f850a2",
   "metadata": {},
   "source": [
    "## 3. Summary & Takeaways\n",
    "- **Item-Based CF** achieves significantly lower prediction error (lower RMSE and MAE) because item similarity patterns are dense and robust across users.\n",
    "- **User-Based CF** exhibits higher catalog coverage because users possess diverse idiosyncratic tastes that surface a wider variety of recommendations.\n"
   ]
  }
 ],
 "metadata": {},
 "nbformat": 4,
 "nbformat_minor": 5
}
